# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)

# Show the metadata information (name/description)
md = dataset.metadata
print(f"{md.name}: {md.description}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

In [ ]:
# List available record sets and their fields by @id
if hasattr(dataset, 'record_sets'):
    record_sets = dataset.record_sets
else:
    # fallback for older mlcroissant versions
    record_sets = dataset.metadata.record_sets

print('Available Record Sets and their fields:')
record_set_ids = []
for rs in record_sets:
    print(f"\nRecordSet name: {rs.name}")
    print(f"RecordSet @id: {rs.id}")
    record_set_ids.append(rs.id)
    if hasattr(rs, 'fields') and rs.fields:
        print('  Fields:')
        for field in rs.fields:
            print(f"    - {field.name} (@id: {field.id}, dataType: {field.data_type})")
    else:
        print('  No fields found.')

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.


In [ ]:
# Extract data from all available record sets
# This uses their @id as reference and stores DataFrames per record set.

dataframes = {}
for record_set_id in record_set_ids:
    print(f"Loading records for RecordSet: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    if len(records) == 0:
        print(f"  [!] No records found for {record_set_id}")
        continue
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"  Loaded {len(df)} records. Columns (@id):\n  {list(df.columns)}\n")
if dataframes:
    # Select the first one as the main for exploration
    main_record_set_id = list(dataframes.keys())[0]
    print(f"Selected main record set for EDA: {main_record_set_id}")
    print(f"Column (field) @ids in this set: {dataframes[main_record_set_id].columns.tolist()}")
    display(dataframes[main_record_set_id].head())
else:
    print('No dataframes loaded! Check dataset schema or Croissant source.')

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section demonstrates removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.


In [ ]:
import numpy as np

# For EDA, select a numeric field from columns
df = dataframes[main_record_set_id]
numeric_candidates = []
for col in df.columns:
    # Heuristic: try float conversion for a sample
    try:
        _ = df[col].dropna().astype(float)
        numeric_candidates.append(col)
    except:
        continue

if numeric_candidates:
    numeric_field_id = numeric_candidates[0]  # Pick the first numeric column by @id
    print(f"Selected numeric field for filtering: {numeric_field_id}")
else:
    print('No numeric field available for EDA!')
    numeric_field_id = None

if numeric_field_id is not None:
    df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
    threshold = df[numeric_field_id].quantile(0.75)  # e.g., upper quartile
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
    print(filtered_df[[numeric_field_id]].head())

    # Normalization
    mean = filtered_df[numeric_field_id].mean()
    std = filtered_df[numeric_field_id].std()
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - mean) / std
    print(f"\nNormalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Try grouping by a likely categorical column
    candidate_group_fields = [col for col in df.columns if col != numeric_field_id]
    group_field = None
    for col in candidate_group_fields:
        if df[col].dtype == object and df[col].nunique() < 10:
            group_field = col
            break

    if group_field:
        grouped_df = (
            filtered_df.groupby(group_field)[numeric_field_id].mean().to_frame(f"mean_{numeric_field_id}")
        )
        print(f"\nGrouped data by '{group_field}' (showing average {numeric_field_id}):")
        print(grouped_df.head())
    else:
        print("No suitable group field found.")
else:
    print("No numeric EDA performed.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
# Visualize the distribution and relationships of numeric fields
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style='whitegrid')

if numeric_field_id is not None and numeric_field_id in df.columns:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field_id].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

    # Pairplot with another numeric field if available
    if len(numeric_candidates) > 1:
        second_field = numeric_candidates[1]
        plt.figure(figsize=(6,6))
        sns.scatterplot(
            x=df[numeric_field_id],
            y=df[second_field]
        )
        plt.xlabel(numeric_field_id)
        plt.ylabel(second_field)
        plt.title(f"Scatter of {numeric_field_id} vs {second_field}")
        plt.show()
else:
    print("No numeric field available for visualization.")

## 6. Conclusion
This notebook demonstrated the loading and basic exploration of the FAIR² dataset using `mlcroissant`.

Key steps performed:
- Loaded metadata and record sets via schema URL
- Inspected record sets and field `@id` values
- Loaded record sets into DataFrames and listed available columns
- Performed exploratory analysis with numeric fields, including filtering and normalization
- Produced some basic visualizations

For deeper analysis or domain-specific approaches, consult the dataset documentation, inspect the full schema, and visit [https://sen.science/doi/10.71728/senscience.vq4a-28xa](https://sen.science/doi/10.71728/senscience.vq4a-28xa) for more resources.